In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
from pathlib import Path

# Log-Verzeichnis
LOG_DIR = Path(cfg.paths.logs_dir)

# Verzeichnis anlegen, wenn nicht vorhanden
LOG_DIR.mkdir(exist_ok=True)

print(f"Log-Verzeichnis: {LOG_DIR.resolve()}")

In [ ]:
import subprocess
from pathlib import Path
from datetime import datetime

def run_notebook(path):
    nb_path = Path(path)
    nb_name = nb_path.stem
    log_file = LOG_DIR / f"{nb_name}.log"

    print(f"\nStarte: {path}")
    print(f"Logfile: {log_file.name}")

    cmd = [
        "papermill",
        str(nb_path),
        str(nb_path),     # inplace
        "--log-output"
    ]

    with open(log_file, "a", encoding="utf-8") as lf:
        lf.write(f"\n=== {datetime.now()} | START {path} ===\n")

        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            print(line, end="")
            lf.write(line)

        exit_code = process.wait()

        if exit_code != 0:
            raise RuntimeError(f"Fehler in {path}")

        lf.write(f"=== {datetime.now()} | END {path} ===\n")
        print(f"Erledigt: {path}")

# Pipeline
pipeline = cfg.pipeline.notebooks

for nb in pipeline:
    run_notebook(nb)

print("\nGesamte Pipeline erfolgreich durchlaufen.")